**Task-Specific Classes**
- to tackle our binary classification task using BERT, we can use the BertForSequenceClassification class provided by the transformers libary/ It is just a bert model plus a classification head on top of it. All thats needed is to specify the bert checkpoint to use, the number of output units for the classification task and optionally the data type (16-bit floats to be used to fit the model on small GPUs)

In [2]:
import transformers
from datasets import load_dataset
import torch
import torch.nn as nn 
import tokenizers

In [9]:
from transformers import BertForSequenceClassification

In [3]:
imdb_dataset = load_dataset("imdb")
split = imdb_dataset['train'].train_test_split(train_size=0.8, seed=42)
imdb_train_set, imdb_valid_set = split['train'], split['test']
imdb_test_set = imdb_dataset['test']

In [4]:
bert_tokenizer = transformers.AutoTokenizer.from_pretrained("bert-base-uncased")

In [5]:
torch.manual_seed(42)

In [7]:
device = torch.device("mps" if torch.mps.is_available() else "gpu")

In [10]:
bert_for_binary_clf = BertForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=2, dtype=torch.float16).to(device)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


The transformers library contains many task-specific classes based on various pretrained models such as BertForQuestionAnswering or RobertaForSequenceClassification. Can also use AutoModelForSequenceClassification to let the library pick the right class for you, based on the requested model checkpoint

always used a single output for binary classification but set num_labels to 2 when setting up bert for sequence classification.... why??
- for simplicity hugging face prefers to treat binary classification exactly like multiclass classification - with the model outputting 2 logits instead of 1 and must be trained with crossentropyloss 

In [13]:
encoding = bert_tokenizer(["This was a great movie!"], return_tensors='pt').to(device)
with torch.no_grad():
    output = bert_for_binary_clf(
        **encoding
    )
output.logits

tensor([[-0.0122,  0.6313]], device='mps:0', dtype=torch.float16)

In [14]:
torch.softmax(output.logits, dim=-1)

tensor([[0.3445, 0.6558]], device='mps:0', dtype=torch.float16)

first, we tokenize the review, then we call the model, it returns an output object containing the logits and we convert these logits to estimated probabilities using the torch.softmax() function. Important to always remember that the classification head of the model is untrained. 
- if you pass labels when calling this model (or any other model from the transformers library), then it also computes the loss and returns it in the ModelOutput object. For example:

In [15]:
with torch.no_grad():
    output = bert_for_binary_clf(
        **encoding,
        labels=torch.tensor([1], device=device)
    )
output.loss

tensor(0.4224, device='mps:0', dtype=torch.float16)

every model from the transformers library has a config attribute that contains the model's configuration....

In [16]:
bert_for_binary_clf.config

BertConfig {
  "architectures": [
    "BertForMaskedLM"
  ],
  "attention_probs_dropout_prob": 0.1,
  "classifier_dropout": null,
  "dtype": "float16",
  "gradient_checkpointing": false,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 768,
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "layer_norm_eps": 1e-12,
  "max_position_embeddings": 512,
  "model_type": "bert",
  "num_attention_heads": 12,
  "num_hidden_layers": 12,
  "pad_token_id": 0,
  "position_embedding_type": "absolute",
  "problem_type": "single_label_classification",
  "transformers_version": "4.56.2",
  "type_vocab_size": 2,
  "use_cache": true,
  "vocab_size": 30522
}

passing num labels as 1 results in the model using mse loss. but > 1 results in cross entropy. We could train this model using our own trainer but the transformers library provides a convenient Trainer API..

In [17]:
bert_for_binary_clf

BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e

**The Trainer API:**
- The Trainer API lets you fine-tune a model on your own dataset with little boilerplate code. saves model checkpoints during training..., applies early stopping..., distributes the computations across GPUs, log metrics, take care of padding, batching, and shuffling, and more. Lets use the Trainer API to train the IMDB model/
- The trainer API works directly with dataset objects, not data loaders, but it expects the datasets to contain tokenized text not strings. we can use the dataset map method to do this..

In [18]:
def tokenize_batch(batch):
    return bert_tokenizer(batch['text'], truncation=True, max_length=200)

In [19]:
tok_imdb_train_set = imdb_train_set.map(tokenize_batch, batched=True)
tok_imdb_valid_set = imdb_valid_set.map(tokenize_batch, batched=True)
tok_imdb_test_set = imdb_test_set.map(tokenize_batch, batched=True)

Map:   0%|          | 0/20000 [00:00<?, ? examples/s]

Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

The tokenize_batch() method tokenizes the given batch of reviews and the resulting fields are added to each instance by the map() method. This includes fields such as token_ids and attention_mask which the model expects..

In [20]:
tok_imdb_train_set

Dataset({
    features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 20000
})

as seen above, the tokenizer extra arguments are added to the data....

To evaluate the model, we can write a simple function that takes an object with 2 attributes label_ids and predictions:

In [21]:
def compute_accuracy(pred):
    return {"accuracy": (pred.label_ids == pred.predictions.argmax(-1)).mean()}

alternatively, we could use the metrics provided by the huggingface evaluate library

next, we must specify our training config in a TrainingArguments object:

In [22]:
from transformers import TrainingArguments

In [ ]:
train_args = TrainingArguments(
    output_dir = "my_imdb_model",
    num_train_epochs = 2,
    per_device_train_batch_size = 128,
    per_device_eval_batch_size = 128,
    eval_strategy="epoch",
    logging_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    report_to="none"
)